__!!! Activate smplx/.conda environment before running this notebook.__  
Won't run on Headless server (e.g. HPC) because it requires GUI.

### 0. Initialization

In [ ]:
# Import libraries
import sys
import torch
import smplx
import trimesh
import pyrender
import numpy as np
import random
import pickle
import os
import matplotlib.pyplot as plt
import h5py

from torch.utils.data import DataLoader
from datasets import HDF5Dataset
from utils import visualize_smpl_with_joints, plot_input_channels, get_preprocessed_hdf5_path

np.set_printoptions(threshold=sys.maxsize, precision=2, suppress=True)
torch.set_printoptions(precision=4, sci_mode=False)
# %matplotlib inline	# Uncomment this line if you are using Jupyter Notebook

# Path to SMPL models
smpl_feml_model_path_v1_0 = '/home/nadeemshah/coding/bodies-at-rest/smpl/models/basicModel_f_lbs_10_207_0_v1.0.0.pkl'	# v1.0.0 has only 10 shape coefficients

# Load SMPL models
model = smplx.SMPL(smpl_feml_model_path_v1_0)

# HDF5 file paths
# hdf5_file_name = 'preprocessed_mod1_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_1__normalize_per_image_True.hdf5'
# hdf5_file_name = 'preprocessed_mod2_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_2__normalize_per_image_True.hdf5'
# hdf5_file_name = 'preprocessed_straight_limbs_mod1_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_1__normalize_per_image_True_no_75mm.hdf5'
hdf5_file_name = 'preprocessed_straight_limbs_mod1_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_1__normalize_per_image_True.hdf5'

hdf5_file_path = get_preprocessed_hdf5_path(hdf5_file_name)

### Visualization of input channels & SMPL joints overlaid on the mesh

In [ ]:
index = 0

### Load the HDF5 file ###
with h5py.File(hdf5_file_path, 'r') as f:
	inputs = f['train/straight_limbs/f/inputs'][index]
	print(f"inputs_hdf5:		{inputs.shape}, dtype: {inputs.dtype}, type: {type(inputs)}")
	inputs = torch.tensor(inputs).unsqueeze(0)
	labels = f['train/straight_limbs/f/labels'][index]
	print(f"labels_hdf5:		{labels.shape}, dtype: {labels.dtype}, type: {type(labels)}")
	labels = torch.tensor(labels).unsqueeze(0)
	print(f"inputs_hdf5:		{inputs.shape}, dtype: {inputs.dtype}, type: {type(inputs)}")
	print(f"labels_hdf5:		{labels.shape}, dtype: {labels.dtype}, type: {type(labels)}")
	# plot_input_channels(inputs, index)

# Extract SMPL parameters from true_labels
joint_positions	= labels[:, 0:72].view(-1, 24, 3)	# Shape: (batch_size, 24, 3)
betas			= labels[:, 72:82]     # Shape: (batch_size, 10)
global_orient	= labels[:, 82:85]     # Shape: (batch_size, 3)
body_pose		= labels[:, 85:154]    # Shape: (batch_size, 69)
transl			= labels[:, 154:157]   # Shape: (batch_size, 3)

# Print shapes and dtypes of the extracted parameters
print()
print(f"joint_positions:	{joint_positions.shape}, dtype: {joint_positions.dtype}")
print(f"betas:			{betas.shape}, dtype: {betas.dtype}")
print(f"global_orient:		{global_orient.shape}, dtype: {global_orient.dtype}")
print(f"body_pose:		{body_pose.shape}, dtype: {body_pose.dtype}")
print(f"transl:			{transl.shape}, dtype: {transl.dtype}")

# Visualize the SMPL model with joints
visualize_smpl_with_joints(
    model			= model,
    body_pose_1     = body_pose,
    global_orient_1 = global_orient,
    betas_1         = betas,
    transl_1        = transl,
    joints_1        = joint_positions[0],
	# show_ground     = False,
	joint_radius    = 0.05,  # Increased from 0.015 to 0.05 for better visibility
)

### Calculate & print input channel statistics

In [ ]:
def print_input_stats_before_and_after_normalization(input_tensor, caption=""):
	print(f"{caption} - Input Tensor Statistics:", end="\n\n")
	print(f"input_tensor (before):	{input_tensor.shape}, dtype: {input_tensor.dtype}")
	print(f"Overall ->	min: {input_tensor.min():.4f}, max: {input_tensor.max():.4f}, mean: {input_tensor.mean():.4f}, std: {input_tensor.std():.4f}")
	print(f"Channel 0 ->	min: {input_tensor[:, 0].min():.4f}, max: {input_tensor[:, 0].max():.4f}, mean: {input_tensor[:, 0].mean():.4f}, std: {input_tensor[:, 0].std():.4f}")
	print(f"Channel 1 ->	min: {input_tensor[:, 1].min():.4f}, max: {input_tensor[:, 1].max():.4f}, mean: {input_tensor[:, 1].mean():.4f}, std: {input_tensor[:, 1].std():.4f}")
	print(f"Channel 2 ->	min: {input_tensor[:, 2].min():.4f}, max: {input_tensor[:, 2].max():.4f}, mean: {input_tensor[:, 2].mean():.4f}, std: {input_tensor[:, 2].std():.4f}")

	# Apply means and standard deviations if provided
	means = torch.tensor([26.201084, 11.778635, 11.731706], dtype=torch.float32).view(-1, 1, 1)
	std_devs = torch.tensor([41.360558, 27.982226, 8.824089], dtype=torch.float32).view(-1, 1, 1)
	input_tensor = (input_tensor - means) / std_devs

	print()
	print(f"input_tensor (after):	{input_tensor.shape}, dtype: {input_tensor.dtype}")
	print(f"Overall ->	min: {input_tensor.min():.4f}, max: {input_tensor.max():.4f}, mean: {input_tensor.mean():.4f}, std: {input_tensor.std():.4f}")
	print(f"Channel 0 ->	min: {input_tensor[:, 0].min():.4f}, max: {input_tensor[:, 0].max():.4f}, mean: {input_tensor[:, 0].mean():.4f}, std: {input_tensor[:, 0].std():.4f}")
	print(f"Channel 1 ->	min: {input_tensor[:, 1].min():.4f}, max: {input_tensor[:, 1].max():.4f}, mean: {input_tensor[:, 1].mean():.4f}, std: {input_tensor[:, 1].std():.4f}")
	print(f"Channel 2 ->	min: {input_tensor[:, 2].min():.4f}, max: {input_tensor[:, 2].max():.4f}, mean: {input_tensor[:, 2].mean():.4f}, std: {input_tensor[:, 2].std():.4f}")

In [ ]:
with h5py.File(hdf5_file_path, 'r') as hdf5_file:
	inputs = hdf5_file['train/straight_limbs/f/inputs'][:]
	labels = hdf5_file['train/straight_limbs/f/labels'][:]

	input_tensor = torch.tensor(inputs, dtype=torch.float32)
	label_tensor = torch.tensor(labels, dtype=torch.float32)

print_input_stats_before_and_after_normalization(input_tensor, caption="Female Only")

In [ ]:
with h5py.File(hdf5_file_path, 'r') as hdf5_file:
	inputs = hdf5_file['train/straight_limbs/m/inputs'][:]
	labels = hdf5_file['train/straight_limbs/m/labels'][:]

	input_tensor = torch.tensor(inputs, dtype=torch.float32)
	label_tensor = torch.tensor(labels, dtype=torch.float32)

print_input_stats_before_and_after_normalization(input_tensor, caption="Male Only")

In [ ]:
with h5py.File(hdf5_file_path, 'r') as hdf5_file:
	inputs_f = hdf5_file['train/straight_limbs/f/inputs'][:]
	labels_f = hdf5_file['train/straight_limbs/f/labels'][:]

	inputs_m = hdf5_file['train/straight_limbs/m/inputs'][:]
	labels_m = hdf5_file['train/straight_limbs/m/labels'][:]

	# Concatenate the inputs and labels
	inputs = np.concatenate((inputs_f, inputs_m), axis=0)
	labels = np.concatenate((labels_f, labels_m), axis=0)

	input_tensor = torch.tensor(inputs, dtype=torch.float32)
	label_tensor = torch.tensor(labels, dtype=torch.float32)

print_input_stats_before_and_after_normalization(input_tensor, caption="Combined (Female + Male)")